# 2.05 Ollama Embeddings (로컬)

`OllamaEmbeddings`는 로컬 Ollama 데몬으로 `nomic-embed-text`, `mxbai-embed-large` 같은 오픈소스 임베더를 호출한다.
**완전 오프라인**, **데이터 외부 전송 없음**이 핵심 가치다. GPU 없이도 CPU에서 실용 속도로 동작한다.

## 학습 목표

- Ollama 데몬 기반으로 `OllamaEmbeddings`를 호출한다
- `nomic-embed-text` (768d) 기본값과 다른 로컬 모델 선택지를 비교한다
- `num_ctx`, `keep_alive`로 메모리·레이턴시를 튜닝한다
- 로컬 Chroma에 붙여 완전 오프라인 RAG를 구성한다

## 언제 쓰나

- **민감 데이터** (사내 문서, 개인정보, 의료/법률)가 외부 API로 나가면 안 될 때
- **오프라인 개발**, **에어갭 환경**, **비용 제로 PoC**
- **다른 로컬 LLM**(Ollama ChatOllama)과 같은 런타임으로 묶어서 관리

## 2.05.1 환경 설정

사전 준비:
1. Ollama 설치: https://ollama.com/download
2. 데몬 실행: `ollama serve` (macOS 앱은 자동 실행)
3. 모델 내려받기: `ollama pull nomic-embed-text`

환경 변수가 아닌 **데몬 가동 여부**가 진입 조건. 아래 probe 셀이 실제 호출로 확인한다.

In [ ]:
# !pip install -U langchain langchain-ollama

from langchain_ollama import OllamaEmbeddings
OllamaEmbeddings(model="nomic-embed-text").embed_query("ping")  # daemon check

## 2.05.2 기본 사용법

`embed_query` / `embed_documents`는 관리형 공급자와 완전히 동일한 인터페이스.

In [ ]:
embeddings = OllamaEmbeddings(model="nomic-embed-text")

q_vec = embeddings.embed_query("로컬 임베딩의 장점은?")
print("query dim:", len(q_vec), "| preview:", q_vec[:4])

docs = [
    "Ollama는 로컬에서 LLM을 실행하는 런타임이다.",
    "nomic-embed-text는 768차원 임베딩을 생성한다.",
    "오프라인 환경에서도 RAG를 구성할 수 있다.",
]
doc_vecs = embeddings.embed_documents(docs)
print("docs:", len(doc_vecs), "x", len(doc_vecs[0]))

## 2.05.3 차원 · 비용 · 다국어 성능

| 모델 | 차원 | 크기 | 언어 | 메모 |
|------|------|------|------|------|
| `nomic-embed-text` | 768 | ~275MB | 영어 중심 | 기본 추천, MTEB 상위권 |
| `mxbai-embed-large` | 1024 | ~670MB | 영어 중심 | 더 높은 품질, 더 느림 |
| `bge-m3` (별도 pull) | 1024 | ~2.3GB | 100+ 다국어 | 한국어 강함 |
| `snowflake-arctic-embed` | 1024 | ~670MB | 영어 | Snowflake 사내 |

- **비용**: 전력/시간만 — API 비용 없음
- **한국어**: `nomic-embed-text`는 영어 중심. 한국어 RAG는 `bge-m3` 또는 HuggingFace 경로(06 노트북) 권장
- **속도**: M-시리즈 Mac CPU에서 수십~수백 문장/초

## 2.05.4 벡터스토어 연동 미리보기

로컬 임베더 + 로컬 Chroma = 완전 오프라인 RAG.

In [ ]:
# !pip install -U langchain-chroma
from langchain_chroma import Chroma

store = Chroma.from_texts(
    texts=docs,
    embedding=OllamaEmbeddings(model="nomic-embed-text"),
    collection_name="demo_ollama",
)
hits = store.similarity_search("오프라인 RAG 구성", k=2)
for h in hits:
    print("-", h.page_content)

## 2.05.5 로컬 튜닝 — `num_ctx`, `keep_alive`, `num_thread`

| 파라미터 | 의미 | 권장 |
|----------|------|------|
| `num_ctx` | 단일 호출 최대 토큰 | 기본(`2048`) 또는 모델 한도까지 |
| `keep_alive` | 모델을 메모리에 유지하는 초 (`int`, `-1`은 무기한) | 반복 호출 시 `3600`(1시간) |
| `num_thread` | CPU 스레드 수 | 물리 코어 수 |
| `base_url` | 원격 Ollama 서버 | `http://<host>:11434` |

> LangChain v1의 `OllamaEmbeddings`는 `keep_alive`를 **정수(초)** 로 받는다. `"5m"`, `"1h"` 같은 문자열은 REST API 레벨에서만 허용되며 래퍼에서는 초 단위로 환산해 넘긴다.

In [ ]:
tuned = OllamaEmbeddings(
    model="nomic-embed-text",
    num_ctx=2048,
    keep_alive=3600,     # 1시간 동안 메모리에 유지 → 반복 호출 콜드 스타트 제거
)
v = tuned.embed_query("튜닝된 로컬 임베더")
print("tuned dim:", len(v))

## 2.05.6 원격 Ollama 서버

팀 내 GPU 서버에 Ollama를 띄우고 노트북에서 `base_url`만 지정해 쓰는 패턴.

In [ ]:
# 원격 Ollama 서버 (예시)
# remote = OllamaEmbeddings(
#     model="nomic-embed-text",
#     base_url="http://gpu-server.internal:11434",
# )
# print(remote.embed_query("원격 호출 테스트")[:3])
print("원격 섹션은 예시. 사내 GPU 서버 주소를 지정해서 실행하라.")

## 체크리스트

- [ ] `ollama serve` 데몬이 가동 중이고 `ollama pull`로 모델을 미리 받았는가
- [ ] 한국어 코퍼스라면 `bge-m3` 또는 HuggingFace 경로(06) 고려
- [ ] 반복 적재 잡은 `keep_alive=3600`(초)로 콜드 스타트 제거
- [ ] 원격 서버를 쓴다면 `base_url`과 네트워크/방화벽 확인

## 다음

- `06_huggingface.ipynb` — Sentence Transformers 직접 경로와 `BAAI/bge-m3`
- `../03_vectorstores/` — 로컬 Chroma를 디스크 영구 저장으로 설정

## 참고

- 공식: https://docs.langchain.com/oss/python/integrations/text_embedding/ollama
- Ollama 모델 허브: https://ollama.com/library
- nomic-embed-text: https://ollama.com/library/nomic-embed-text